# Notebook 02 – Silver Layer

**Why Silver Layer?**

The Silver layer is responsible for:

- Extracting healthcare resources
- Cleaning raw data
- Standardizing schemas
- Merging multiple healthcare formats
- Removing duplicates
- Preparing analytics-ready datasets

Unlike the Bronze layer, transformations and business logic are applied here.

**Objective**
The objective of this notebook is to transform raw healthcare data from the Bronze layer into clean, standardized, and analytics-ready datasets.

In this layer, data from multiple healthcare formats (FHIR, HL7, and Flat Files) is extracted, cleaned, standardized, and merged into unified Silver tables.

**Input Tables**

This notebook reads data from the Bronze layer:

- bronze.fhir_raw
- bronze.hl7_adt_raw
- bronze.patients_raw
- bronze.encounters_raw
- bronze.conditions_raw
- bronze.observations_raw
- bronze.medications_raw

### 1. Read Bronze FHIR Table

In [0]:
bronze_df = spark.read.table("bronze.fhir_raw")

In [0]:
bronze_df.printSchema()

Reads the raw FHIR JSON data stored in the Bronze layer.

The schema is inspected to understand the nested FHIR structure before extraction.

### 2. Explode FHIR Bundle

FHIR stores healthcare resources inside an array called **entry**.

**Code Used**

explode("entry")

Each FHIR Bundle contains multiple healthcare resources. By exploding the **`entry`** array, every resource is extracted into an individual row for further processing.

#### Before Explode

FHIR Bundle

⬇️
Bundle  
├── Patient  
├── Encounter  
├── Condition  
├── Observation  
└── MedicationRequest

#### After Applying `explode("entry")`

- Patient
- Encounter
- Condition
- Observation
- MedicationRequest
...

Each healthcare resource can now be filtered, extracted, and transformed independently in the Silver layer.

In [0]:
from pyspark.sql.functions import explode

In [0]:
resource_df = bronze_df.select(explode("entry").alias("entry"))

### 3. Patient Resource

Filters: resourceType == "Patient"

Extracts fields such as

- Patient ID
- Name
- Gender
- Birth Date
- Address
- Phone Number

Creates: patient_fhir_df

In [0]:
patient_df = resource_df.filter(resource_df["entry.resource.resourceType"] == 'Patient')

In [0]:
from pyspark.sql.functions import col, get_json_object, concat_ws, lit

patient_fhir_df = (
    patient_df.select(
        col("entry.resource.id").alias("patient_id"),
        col("entry.resource.birthDate").alias("birth_date"),
        col("entry.resource.gender").alias("gender"),
        col("entry.resource.name").alias("name")
    )
    .withColumn(
        "given_name",
        get_json_object("name", "$[0].given[0]")
    )
    .withColumn(
        "family_name",
        get_json_object("name", "$[0].family")
    )
    .withColumn(
        "full_name",
        concat_ws(" ", col("given_name"), col("family_name"))
    )
    .withColumn(
        "address",
        lit(None).cast("string")
    )
    .withColumn(
        "phone",
        lit(None).cast("string")
    )
    .select(
        "patient_id",
        "full_name",
        "birth_date",
        "gender",
        "address",
        "phone"
    )
)

display(patient_fhir_df)

###  4. Encourter Records ### 
Filters: resourceType == "Encounter"

Extracts

- Encounter ID
- Patient Reference
- Start Date
- End Date
- Class
- Organization
- Provider
- Reason

Creates: encounter_fhir_df

In [0]:
encounter_df = resource_df.filter(resource_df["entry.resource.resourceType"] == 'Encounter')

In [0]:
from pyspark.sql.functions import col, lit

encounter_fhir_df = (
    encounter_df.select(
        col("entry.resource.id").alias("encounter_id"),
        col("entry.resource.subject.reference").alias("patient_reference"),
        col("entry.resource.period.start").alias("start_date"),
        col("entry.resource.period.end").alias("end_date"),
        col("entry.resource.class.code").alias("patient_class"),
        lit(None).cast("string").alias("provider"),
        lit(None).cast("string").alias("organization"),
        col("entry.resource.status").alias("reason"),
        lit(None).cast("string").alias("reason_description")
    )
)

In [0]:
display(encounter_fhir_df)

### 5. Condition Resource### 
Filters: resourceType == "Condition"

Extracts

- Condition ID
- Clinical Status
- Verification Status
- Diagnosis Code
- Diagnosis Name
- Patient Reference
- Encounter Reference
- Recorded Date

Creates: condition_fhir_df

In [0]:
from pyspark.sql.functions import get_json_object

In [0]:
condition_df = resource_df.filter(resource_df["entry.resource.resourceType"]=="Condition")

condition_fhir_df = condition_df.select(
    col("entry.resource.id").alias("Condition_Id"),
    col("entry.resource.clinicalStatus.coding.code")[0].alias("Clinical_Status"),
    col("entry.resource.verificationStatus.coding.code")[0].alias("Verification_Status"),
    get_json_object(
        col("entry.resource.code"),
        "$.coding[0].code"
    ).alias("Condition_Code"),

   
    get_json_object(
        col("entry.resource.code"),
        "$.coding[0].display"
    ).alias("Condition_Name"),

    get_json_object(
        col("entry.resource.category")[0],
        "$.coding[0].display"
    ).alias("Category"),
   
    col("entry.resource.subject.reference").alias("Patient_Reference"),
    col("entry.resource.encounter.reference").alias("Encounter_Reference"),
    col("entry.resource.recordedDate").alias("Recorded_Date"),
    col("entry.resource.onsetDateTime").alias("Onset_Date")
)

display(condition_fhir_df)



### 6. Observation Resource### 

Filters: resourceType == "Observation"

Extracts

- Observation ID
- Observation Name
- Value
- Unit
- Observation Date
- Patient Reference
- Encounter Reference

Creates: observation_fhir_df

In [0]:
observation_df = resource_df.filter(resource_df["entry.resource.resourceType"]=="Observation")



In [0]:
from pyspark.sql.functions import to_date

observation_fhir_df = observation_df.select(
    col("entry.resource.id").alias("Observation_Id"),
    col("entry.resource.status").alias("Status"),

    get_json_object(
        col("entry.resource.category")[0],
        "$.coding[0].display"
    ).alias("Observation_Name"),

    col("entry.resource.subject.reference").alias("Patient_Reference"),
    col("entry.resource.encounter.reference").alias("Encounter_Reference"),
    to_date(col("entry.resource.effectiveDateTime")).alias("Observation_Date"),
    to_date(col("entry.resource.issued")).alias("Issued_Date"),
    col("entry.resource.valueQuantity.value").cast("string").alias("Value"),
    col("entry.resource.valueQuantity.unit").alias("Unit")
)

display(observation_fhir_df)

### 7. MedicationRequest Resource
### 
Filters: resourceType == "MedicationRequest"

Extracts

- Medication ID
- Medication Name
- Patient Reference
- Encounter Reference
- Status
- Intent

Creates: medicationRequest_fhir_df

In [0]:
medicationRequest_df = resource_df.filter(resource_df["entry.resource.resourceType"]=="MedicationRequest")

In [0]:
medicationRequest_fhir_df = medicationRequest_df.select(
    col("entry.resource.id").alias("MedicationRequest_Id"),
    col("entry.resource.status").alias("Status"),
    col("entry.resource.intent").alias("Intent"),
    col("entry.resource.subject.reference").alias("Patient_Reference"),
    col("entry.resource.medicationCodeableConcept.coding.code")[0].alias("Medication_Code"),
    col("entry.resource.medicationCodeableConcept.coding.display")[0].alias("Medication_Name"),
    col("entry.resource.encounter.reference").alias("Encounter_Reference"),
    col("entry.resource.authoredOn").alias("Authored_On"),
    col("entry.resource.requester.reference").alias("Requester_Reference"),
    col("entry.resource.requester.display").alias("Requester_Name"),
    
)


display(medicationRequest_fhir_df)

## **HL7 Processing**

9. Read HL7 Bronze Table

In [0]:
hl7_df = spark.read.table("bronze.hl7_adt_raw")

Reads raw HL7 ADT messages.

In [0]:
hl7_df.printSchema()

In [0]:
display(hl7_df)

### 10. Split HL7 Segments
### 
HL7 messages are divided into segments such as

PID
PV1

In [0]:
from pyspark.sql.functions import split, explode

hl7_segments = hl7_df.withColumn(
    "segments",
    split("raw_message", "\n")
)

segment_df = hl7_segments.select(
    "path",
    explode("segments").alias("segment")
)

In [0]:
from pyspark.sql.functions import split

segment_df = segment_df.withColumn(
    "segment_type",
    split("segment", "\\|").getItem(0)
)

This converts raw HL7 text into structured fields.

In [0]:
patient_segment = segment_df.filter("segment_type='PID'")

### 11. Patient HL7### 
From the PID segment, the notebook extracts

- Patient ID
- Name
- Gender
- Birth Date

In [0]:
from pyspark.sql.functions import split

patient_hl7_df = patient_segment.withColumn(
    "fields",
    split("segment", "\\|")
)

In [0]:
patient_hl7_df.select("fields").show(1, truncate=False)

In [0]:
from pyspark.sql.functions import col, split, concat_ws, get

patient_hl7_df = (
    patient_segment
    .withColumn("fields", split(col("segment"), "\\|"))
    .select(
        get(col("fields"), 3).alias("patient_id"),
        get(col("fields"), 5).alias("patient_name"),
        to_date(get(col("fields"), 7), "yyyyMMdd").alias("birth_date"),
        get(col("fields"), 8).alias("gender"),
        get(col("fields"), 11).alias("address"),
        get(col("fields"), 13).alias("phone")
    )
    .withColumn("name_parts", split(col("patient_name"), "\\^"))
    .withColumn("family_name", get(col("name_parts"), 0))
    .withColumn("given_name", get(col("name_parts"), 1))
    .withColumn("middle_name", get(col("name_parts"), 2))
    .withColumn(
        "full_name",
        concat_ws(" ", col("given_name"), col("middle_name"), col("family_name"))
    )
)

In [0]:
patient_fhir_df.printSchema()

In [0]:
display(patient_hl7_df)

### 12. Encounter HL7

From the PV1 segment, extracts

- Encounter ID
- Patient ID
- Admission Details
- Visit Information


In [0]:
pv1_segment = segment_df.filter(col("segment_type") == "PV1")

In [0]:
encounter_hl7_df = (
    pv1_segment
    .withColumn("fields", split(col("segment"), "\\|"))
)

display(encounter_hl7_df)

In [0]:
encounter_hl7_df = encounter_hl7_df.select(
    get(col("fields"),19).alias("encounter_id"),
    get(col("fields"),2).alias("patient_class"),
    get(col("fields"),3).alias("location"),
    get(col("fields"),7).alias("provider"),
    get(col("fields"),4).alias("admission_type"),
    get(col("fields"),10).alias("hospital_service")
)

In [0]:
display(encounter_hl7_df)

### Flat File Processing
### 
The notebook then reads the Bronze CSV tables.

Examples include

- patients_raw
- encounters_raw
- conditions_raw
- observations_raw
- medications_raw

Columns are renamed and standardized so that they match the FHIR schema.

This allows all sources to be merged later.

In [0]:
patient_flat_df = spark.table("db_healthcare_kl.bronze.patients_raw")

encounter_flat_df = spark.table("db_healthcare_kl.bronze.encounters_raw")

condition_flat_df = spark.table("db_healthcare_kl.bronze.conditions_raw")

observation_flat_df = spark.table("db_healthcare_kl.bronze.observations_raw")

medication_flat_df = spark.table("db_healthcare_kl.bronze.medications_raw")

In [0]:
patient_flat_df.printSchema()

encounter_flat_df.printSchema()

condition_flat_df.printSchema()

observation_flat_df.printSchema()

medication_flat_df.printSchema()

In [0]:
display(patient_flat_df)

display(encounter_flat_df)

display(condition_flat_df)

display(observation_flat_df)

display(medication_flat_df)

In [0]:
from pyspark.sql.functions import concat_ws

patient_flat_df = patient_flat_df.select(
    patient_flat_df.Id.alias("patient_id"),
    concat_ws(" ",
              patient_flat_df.FIRST,
              patient_flat_df.MIDDLE,
              patient_flat_df.LAST).alias("full_name"),
    patient_flat_df.BIRTHDATE.alias("birth_date"),
    patient_flat_df.GENDER.alias("gender"),
    patient_flat_df.ADDRESS.alias("address"),
    patient_flat_df.CITY.alias("city"),
    patient_flat_df.STATE.alias("state"),
    patient_flat_df.ZIP.alias("zip")
)

display(patient_flat_df)

In [0]:
from pyspark.sql.functions import lit

patient_flat_df = (
    patient_flat_df
        .withColumn("phone", lit(None).cast("string"))
        .select(
            "patient_id",
            "full_name",
            "birth_date",
            "gender",
            "address",
            "phone"
        )
)

In [0]:
observation_flat_df = observation_flat_df.select(
    observation_flat_df.PATIENT.alias("patient_id"),
    observation_flat_df.ENCOUNTER.alias("encounter_id"),
    observation_flat_df.CODE.alias("observation_code"),
    observation_flat_df.DESCRIPTION.alias("observation_name"),
    observation_flat_df.VALUE.alias("value"),
    observation_flat_df.UNITS.alias("unit"),
    observation_flat_df.DATE.alias("observation_date")
)

display(observation_flat_df)

In [0]:
medication_flat_df = medication_flat_df.select(
    medication_flat_df.PATIENT.alias("patient_id"),
    medication_flat_df.ENCOUNTER.alias("encounter_id"),
    medication_flat_df.CODE.alias("medication_code"),
    medication_flat_df.DESCRIPTION.alias("medication_name"),
    medication_flat_df.START.alias("start_date"),
    medication_flat_df.STOP.alias("end_date"),
    medication_flat_df.TOTALCOST.alias("total_cost")
)

display(medication_flat_df)

%md
### 3. Union Data from Multiple Healthcare Sources

Healthcare data is received from multiple source systems:

- FHIR
- HL7
- Flat Files

The datasets are combined using **unionByName()**, which merges records by matching column names.

**Example – Patient Data**

FHIR + HL7 + Flat Files

↓

patient_silver_df


The same approach is used for other healthcare entities:

- encounter_silver_df
- condition_silver_df
- observation_silver_df
- medication_silver_df

This creates one unified Silver dataset for each healthcare entity, making downstream analytics and machine learning simpler.

In [0]:
patient_fhir_df = patient_fhir_df.select(
    "patient_id",
    "full_name",
    "birth_date",
    "gender",
    "address",
    "phone"
)

patient_hl7_df = patient_hl7_df.select(
    "patient_id",
    "full_name",
    "birth_date",
    "gender",
    "address",
    "phone"
)

In [0]:
patient_silver_df = (
    patient_fhir_df
    .unionByName(patient_hl7_df)
    .unionByName(patient_flat_df)
)

In [0]:
display(patient_silver_df)

In [0]:
from pyspark.sql.functions import col, lit

encounter_flat_df = (
    encounter_flat_df
    .select(
        col("Id").alias("encounter_id"),
        col("PATIENT").alias("patient_reference"),
        col("START").alias("start_date"),
        col("STOP").alias("end_date"),
        col("ENCOUNTERCLASS").alias("patient_class"),
        col("PROVIDER").alias("provider"),
        col("ORGANIZATION").alias("organization"),
        col("DESCRIPTION").alias("reason"),
        col("REASONDESCRIPTION").alias("reason_description")
    )
)

In [0]:
from pyspark.sql.functions import lit

encounter_hl7_df = (
    encounter_hl7_df
    .withColumn("patient_reference", lit(None))
    .withColumn("start_date", lit(None))
    .withColumn("end_date", lit(None))
    .withColumn("organization", lit(None))
    .withColumn("reason", lit(None))
    .withColumn("reason_description", lit(None))
)

In [0]:
encounter_hl7_df = encounter_hl7_df.select(
    "encounter_id",
    "patient_reference",
    "start_date",
    "end_date",
    "patient_class",
    "provider",
    "organization",
    "reason",
    "reason_description"
)

In [0]:
encounter_fhir_df = encounter_fhir_df.select(
    "encounter_id",
    "patient_reference",
    "start_date",
    "end_date",
    "patient_class",
    "provider",
    "organization",
    "reason",
    "reason_description"
)

In [0]:
encounter_silver_df = (
    encounter_fhir_df
        .unionByName(encounter_hl7_df)
        .unionByName(encounter_flat_df)
)

In [0]:
display(encounter_silver_df)

In [0]:
from pyspark.sql.functions import col, lit

observation_flat_df = (
    observation_flat_df.select(
        col("patient_id").alias("Patient_Reference"),
        col("encounter_id").alias("Encounter_Reference"),
        col("observation_name").alias("Observation_Name"),
        col("observation_date").alias("Observation_Date"),
        col("observation_date").alias("Issued_Date"),
        col("value").alias("Value"),
        col("unit").alias("Unit"),
        lit(None).cast("string").alias("Observation_Id"),
        lit("final").alias("Status")
    )
)

In [0]:
display(observation_flat_df)

In [0]:
observation_silver_df = (
    observation_fhir_df
    .unionByName(observation_flat_df)
)

In [0]:
display(observation_silver_df)

In [0]:
from pyspark.sql.functions import col, lit

condition_flat_df = condition_flat_df.select(
    lit(None).cast("string").alias("Condition_Id"),
    lit(None).cast("string").alias("Clinical_Status"),
    lit(None).cast("string").alias("Verification_Status"),
    col("CODE").alias("Condition_Code"),
    col("DESCRIPTION").alias("Condition_Name"),
    lit(None).cast("string").alias("Category"),
    col("PATIENT").alias("Patient_Reference"),
    col("ENCOUNTER").alias("Encounter_Reference"),
    col("START").alias("Recorded_Date"),
    col("START").alias("Onset_Date")
)

In [0]:
condition_flat_df = condition_flat_df.select(
    "Condition_Id",
    "Clinical_Status",
    "Verification_Status",
    "Condition_Code",
    "Condition_Name",
    "Category",
    "Patient_Reference",
    "Encounter_Reference",
    "Recorded_Date",
    "Onset_Date"
)

In [0]:
print(condition_flat_df.columns)
print(condition_fhir_df.columns)

In [0]:
condition_silver_df = (
    condition_fhir_df
    .unionByName(condition_flat_df)
)

In [0]:
display(condition_silver_df)

In [0]:
from pyspark.sql.functions import col, lit

medication_flat_df = (
    medication_flat_df.select(
        lit(None).cast("string").alias("MedicationRequest_Id"),
        lit(None).cast("string").alias("Status"),
        lit(None).cast("string").alias("Intent"),

        col("patient_id").alias("Patient_Reference"),
        col("medication_code").alias("Medication_Code"),
        col("medication_name").alias("Medication_Name"),
        col("encounter_id").alias("Encounter_Reference"),
        col("start_date").alias("Authored_On"),

        lit(None).cast("string").alias("Requester_Reference"),
        lit(None).cast("string").alias("Requester_Name")
    )
)

In [0]:
medication_flat_df = medication_flat_df.select(
    "MedicationRequest_Id",
    "Status",
    "Intent",
    "Patient_Reference",
    "Medication_Code",
    "Medication_Name",
    "Encounter_Reference",
    "Authored_On",
    "Requester_Reference",
    "Requester_Name"
)

In [0]:
medication_silver_df = (
    medicationRequest_fhir_df
    .unionByName(medication_flat_df)
)

display(medication_silver_df)

In [0]:
display(patient_silver_df)
display(encounter_silver_df)
display(observation_silver_df)
display(condition_silver_df)
display(medication_silver_df)

### Deduplication### 
The notebook removes duplicate records using the resource identifier.

This ensures that duplicate records from multiple sources do not appear in the Silver layer.


In [0]:
patient_fhir_df = patient_fhir_df.dropDuplicates(["Patient_ID"])
encounter_fhir_df = encounter_fhir_df.dropDuplicates(["Encounter_ID"])
condition_fhir_df = condition_fhir_df.dropDuplicates(["Condition_ID"])
observation_fhir_df = observation_fhir_df.dropDuplicates(["Observation_ID"])
medicationRequest_fhir_df = medicationRequest_fhir_df.dropDuplicates(["MedicationRequest_ID"])

### Delta Live Tables (DLT)### 
The final section defines Delta Live Tables (DLT).

DLT is used to:

- Automate Silver table creation
- Apply data quality expectations
- Validate mandatory fields
- Build reliable data pipelines

In [0]:
from pyspark import pipelines as dp

@dp.table(name="patient_silver_dlt")
@dp.expect("valid_patient_id", "patient_id IS NOT NULL")
@dp.expect("valid_full_name", "full_name IS NOT NULL")
@dp.expect("valid_birth_date", "birth_date IS NOT NULL")

def patient_silver():
    return patient_silver_df

@dp.table(name="encounter_silver_dlt")
@dp.expect("valid_encounter_id", "encounter_id IS NOT NULL")
@dp.expect("valid_patient_reference", "patient_reference IS NOT NULL")
@dp.expect("valid_start_date", "start_date IS NOT NULL")

def encounter_silver():
    return encounter_silver_df

@dp.table(name="condition_silver_dlt")
@dp.expect("valid_condition_id", "Condition_Id IS NOT NULL")
@dp.expect("valid_patient_reference", "Patient_Reference IS NOT NULL")
@dp.expect("valid_condition_name", "Condition_Name IS NOT NULL")

def condition_silver():
    return condition_silver_df

@dp.table(name="observation_silver_dlt")
@dp.expect("valid_observation_id", "Observation_Id IS NOT NULL")
@dp.expect("valid_patient_reference", "Patient_Reference IS NOT NULL")
@dp.expect("valid_observation_name", "Observation_Name IS NOT NULL")

def observation_silver():
    return observation_silver_df

@dp.table(name="medication_request_silver_dlt")
@dp.expect("valid_medication_request_id", "MedicationRequest_Id IS NOT NULL")
@dp.expect("valid_patient_reference", "Patient_Reference IS NOT NULL")
@dp.expect("valid_medication_name", "Medication_Name IS NOT NULL")

def medication_request_silver():
    return medication_silver_df

### Output Silver Tables### 
This notebook creates unified Silver tables such as:

- patient_silver
- encounter_silver
- condition_silver
- observation_silver
- medication_silver

These are clean, standardized datasets used by the Gold layer.